In [ ]:
import pandas as pd
import re

# In this part we will prepare our free text notes dataset for modeling

For a recall our dataset structure is the following one :

| note_id | subject_id | hadm_id | charttime | text | source |

- note_id : Unique note identifier
- subject_id : Patient ID
- hadm_id : Hospital admission ID
- charttime : Event timestamp (clinical time)
- text : Clinical note content
- source : indicates whether the note comes from **discharge** or **radiology**


# We will focus on each variable and see how to make it fit for the modeling

# Note_id, subject_id, hadm_id

They will be used to link the different tables and not for the modeling so we need to let these as they are

# Charttime

In [ ]:
# dataset importation

base_path = "../../data/"
df_free_text_notes = pd.read_csv(base_path + "free_text_clinical_notes.csv.gz", compression="gzip")

In [ ]:
df_free_text_notes['charttime'].sample(n=5, random_state=42)

Let's see what type of date we have

In [ ]:
type(df_free_text_notes['charttime'].iloc[0])

We convert them in real date format

In [ ]:
df_free_text_notes["charttime"] = pd.to_datetime(df_free_text_notes["charttime"], errors="coerce")

In [ ]:
df_free_text_notes["charttime"].dtype

Let's see if we have errors

In [ ]:
# vérifion la présence de valeurs nulles dans la colonne 'charttime' après la conversion
df_free_text_notes["charttime"].isnull().sum()

Perfect ! Let's continue by creating useful features related to time that we could use for modeling

In [ ]:
df_free_text_notes["day"] = df_free_text_notes["charttime"].dt.day
df_free_text_notes["year"] = df_free_text_notes["charttime"].dt.year
df_free_text_notes["month"] = df_free_text_notes["charttime"].dt.month
df_free_text_notes["hour"] = df_free_text_notes["charttime"].dt.hour
df_free_text_notes["minute"] = df_free_text_notes["charttime"].dt.minute
df_free_text_notes["second"] = df_free_text_notes["charttime"].dt.second


# Text

We create a function to clean the text

In [ ]:
def clean_text(text):
    text = text.lower()
    text = text.replace("___", " ")
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

We apply it to the text variable

In [ ]:
def clean_text(text):
    text = text.lower()
    text = text.replace("___", " ")
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

batch_size = 10000
clean_texts = []

for i in range(0, len(df_free_text_notes), batch_size):
    batch = df_free_text_notes['text'].iloc[i:i+batch_size]
    cleaned = batch.astype(str).apply(clean_text)
    clean_texts.append(cleaned)

df_free_text_notes['clean_text'] = pd.concat(clean_texts)

Then we apply TF-IDF to tokenise the features and make it ready for modeling

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.9
)

X = vectorizer.fit_transform(df_free_text_notes['clean_text'])

In [ ]:
from scipy import sparse
sparse.save_npz("../../data/X_tfidf.npz", X)

import pickle
with open("../../data/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [ ]:
df_free_text_notes.head()

Our dataset structure is now the following :

note_id | subject_id | hadm_id | source | day | year | month | hour | minute | second | X_tfidf

Let's analyse them

In [ ]:
df_free_text_notes['second'].describe()

The column second contains only 0 so we can delet

Here is the final free text notes dataset

In [ ]:
df_free_text_notes_clean = df_free_text_notes.drop(columns=['charttime', 'second'])

In [ ]:
# on sauvegarde le dataframe nettoyé compressé
df_free_text_notes_clean.to_csv("../../data/df_free_text_notes_clean.csv.gz", index=False, compression="gzip")

In [ ]:
df_free_text_notes_clean.head()

In [ ]:
X[0:20,0:5].toarray()

# Data Standardization

We standarize the source variable

In [ ]:
df_free_text_notes_clean['source'] = df_free_text_notes_clean['source'].map({
    "discharge": "DISCHARGE_SUMMARY",
    "radiology": "RADIOLOGY_REPORT"
})

Mapping to standard concepts (OMOP-like)

In [ ]:
df_free_text_notes_clean['source_concept_id'] = df_free_text_notes_clean['source'].map({
    "DISCHARGE_SUMMARY": 44814645,
    "RADIOLOGY_REPORT": 44814650
})

df_free_text_notes_clean['source_concept_name'] = df_free_text_notes_clean['source'].map({
    "DISCHARGE_SUMMARY": "Discharge summary",
    "RADIOLOGY_REPORT": "Radiology report"
})

We can create now the data dictionary

In [ ]:
data_dict = pd.DataFrame({
    "column": df_free_text_notes_clean.columns,
    "description": [
        "Unique note identifier",
        "Patient identifier",
        "Hospital admission identifier",
        "Type of clinical note (standardized)",
        "Day of clinical event",
        "Year of clinical event",
        "Month of clinical event",
        "Hour of clinical event",
        "Minute of clinical event"
    ]
})

The dataset was cleaned and structured to improve interoperability.

The "source" variable was standardized and mapped to common clinical concepts 
inspired by OMOP vocabulary (concept_id and concept_name).

This allows better integration with external datasets and improves data sharing.

In [ ]:
df_free_text_notes_clean.head()

Final structure :

note_id | subject_id | hadm_id | concept_id | concept_name | day | year | month | hour | minute | X_tfidf

In [ ]:
import pickle
from scipy import sparse

# Importation du TF-IDF
with open("../../data/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

X = sparse.load_npz("../../data/X_tfidf.npz")

# dimensions du TF-IDF
print(X.shape)